# 2025.12 법인 지속거래약화 운영 모델

최종 채택 모델인 `Y_INTERVENE_M12_v2`용 LightGBM과 Isotonic 확률 보정만 사용한다. 2025년 12월 기준 3,372개 법인을 모두 점수화하며 상위 10% 필터는 적용하지 않는다.

In [ ]:
from pathlib import Path
import math
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir() and (path / 'outputs').is_dir())
OUTPUT_DIR = ROOT / 'outputs' / '수익성F_outputs'
MODEL_DIR = ROOT / 'outputs' / '수익성F_models'

TARGET = 'Y_INTERVENE_M12_v2'
FEATURE_SET = 'FS2_R1_DACK_DYNAMIC'
CUTOFF_MONTH = 202512
RANDOM_STATE = 20260718
RETRAIN_FINAL_MODEL = False

DATASET_PATH = OUTPUT_DIR / 'model_m12_intervene_v2_dataset.csv'
FEATURE_SET_PATH = OUTPUT_DIR / 'model_m12_intervene_v2_feature_registry.csv'
OPERATING_FEATURE_PATH = OUTPUT_DIR / 'model_m12_intervene_v2_operating_features_202512.csv'
LOCKED_MODEL_PATH = MODEL_DIR / 'model_m12_intervene_v2_operating_pipeline_202512.joblib'
LOCKED_CALIBRATOR_PATH = MODEL_DIR / 'model_m12_intervene_v2_operating_calibrator_202512.joblib'
WEB_BUNDLE_PATH = MODEL_DIR / 'web_m12_intervene_v2_202512_bundle.joblib'
WEB_SCORE_PATH = ROOT / 'src' / 'models' / 'web_m12_intervene_v2_scores_202512_all_3372.csv'

FINAL_PARAMS = {
    'n_estimators': 220,
    'learning_rate': 0.024068490013144556,
    'num_leaves': 22,
    'max_depth': 6,
    'min_child_samples': 85,
    'subsample': 0.9978564144159624,
    'colsample_bytree': 0.6264619236335867,
    'reg_alpha': 0.001889859045062956,
    'reg_lambda': 9.394600689281587,
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
registry = pd.read_csv(FEATURE_SET_PATH, encoding='utf-8-sig')
features = registry['feature'].tolist()
if len(features) != 56:
    raise RuntimeError(f'Expected 56 features, found {len(features)}')

operating = pd.read_csv(
    OPERATING_FEATURE_PATH,
    encoding='utf-8-sig',
    dtype={'법인ID': 'string'},
    low_memory=False,
)
operating['법인ID'] = operating['법인ID'].astype(str)
if len(operating) != 3372 or operating['법인ID'].nunique() != 3372:
    raise RuntimeError('2025.12 operating data must contain exactly 3,372 firms.')
if operating['cutoff_month'].astype(int).nunique() != 1 or int(operating['cutoff_month'].iloc[0]) != CUTOFF_MONTH:
    raise RuntimeError('Unexpected operating cutoff month.')
missing_features = sorted(set(features) - set(operating.columns))
if missing_features:
    raise KeyError(f'Missing model features: {missing_features}')

print(f'feature_count={len(features):,}')
print(f'scoring_firms={len(operating):,}')
print(f'y_formula_eligible={int(operating["score_eligible"].sum()):,}')
print(f'y_formula_ineligible={int((~operating["score_eligible"].astype(bool)).sum()):,}')

In [ ]:
def build_final_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer(
        [('numeric', SimpleImputer(strategy='median'), slice(0, None))],
        remainder='drop',
    )
    model = LGBMClassifier(
        **FINAL_PARAMS,
        objective='binary',
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )
    return Pipeline([('preprocessor', preprocessor), ('model', model)])


def fit_operating_model(train: pd.DataFrame, feature_names: list[str]):
    y = train[TARGET].to_numpy(dtype=int)
    groups = train['법인ID'].astype(str).to_numpy()
    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE + 101,
    )
    raw_oof = np.full(len(train), np.nan, dtype=float)
    for fit_index, valid_index in cv.split(train[feature_names], y, groups):
        fold_model = build_final_pipeline()
        fold_model.fit(train.iloc[fit_index][feature_names], y[fit_index])
        raw_oof[valid_index] = fold_model.predict_proba(
            train.iloc[valid_index][feature_names]
        )[:, 1]
    if not np.isfinite(raw_oof).all():
        raise RuntimeError('OOF probabilities are incomplete.')
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(raw_oof, y)
    final_model = build_final_pipeline()
    final_model.fit(train[feature_names], y)
    return final_model, calibrator


if RETRAIN_FINAL_MODEL:
    labeled = pd.read_csv(
        DATASET_PATH,
        encoding='utf-8-sig',
        dtype={'법인ID': 'string'},
        low_memory=False,
    )
    labeled['법인ID'] = labeled['법인ID'].astype(str)
    pipeline, calibrator = fit_operating_model(labeled, features)
    joblib.dump(pipeline, LOCKED_MODEL_PATH)
    joblib.dump(calibrator, LOCKED_CALIBRATOR_PATH)
else:
    pipeline = joblib.load(LOCKED_MODEL_PATH)
    calibrator = joblib.load(LOCKED_CALIBRATOR_PATH)

bundle = {
    'pipeline': pipeline,
    'calibrator': calibrator,
    'features': features,
    'target': TARGET,
    'feature_set': FEATURE_SET,
    'cutoff_month': CUTOFF_MONTH,
    'calibration': 'ISOTONIC',
}
joblib.dump(bundle, WEB_BUNDLE_PATH)
print(f'web_bundle={WEB_BUNDLE_PATH}')

In [ ]:
def relative_change_pct(log_difference: pd.Series) -> np.ndarray:
    values = pd.to_numeric(log_difference, errors='coerce').to_numpy(dtype=float)
    return 100.0 * np.expm1(values)


def add_local_shap_top10(frame: pd.DataFrame, result: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    model_pipeline = bundle['pipeline']
    feature_names = bundle['features']
    transformed = model_pipeline.named_steps['preprocessor'].transform(frame[feature_names])
    contributions = model_pipeline.named_steps['model'].predict(
        transformed, pred_contrib=True
    )
    contributions = np.asarray(contributions)[:, :len(feature_names)]
    top_index = np.argsort(-np.abs(contributions), axis=1)[:, :10]
    for rank in range(10):
        index = top_index[:, rank]
        result[f'shap_top{rank + 1}_feature'] = [feature_names[i] for i in index]
        result[f'shap_top{rank + 1}_value'] = contributions[np.arange(len(frame)), index]
    return result


def score_all_firms_202512(frame: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    feature_names = bundle['features']
    missing = sorted(set(feature_names) - set(frame.columns))
    if missing:
        raise KeyError(f'Missing model features: {missing}')
    raw_probability = bundle['pipeline'].predict_proba(frame[feature_names])[:, 1]
    risk_probability = bundle['calibrator'].predict(raw_probability)

    optional_columns = [
        'score_eligible',
        'SEG__baseline_segment_2023',
        'SEG__current_segment',
        'SEG__transition',
        'CTX__업종_대분류__현재',
        'CTX__업종_중분류__현재',
    ]
    result_columns = ['법인ID', 'cutoff_month'] + [
        column for column in optional_columns if column in frame.columns
    ]
    result = frame[result_columns].copy().reset_index(drop=True)
    result['raw_probability'] = raw_probability
    result['risk_probability'] = risk_probability
    result['risk_probability_pct'] = 100.0 * risk_probability
    result['risk_rank_all_3372'] = (
        pd.Series(risk_probability).rank(method='first', ascending=False).astype(int)
    )
    result['risk_percentile'] = 100.0 * (
        len(result) - result['risk_rank_all_3372'] + 1
    ) / len(result)

    for axis, label in {
        'D': '요구불_최근3대이전9_변화율_pct',
        'A': '자동이체_최근3대이전9_변화율_pct',
        'C': '채널_최근3대이전9_변화율_pct',
        'K': '카드_최근3대이전9_변화율_pct',
    }.items():
        result[label] = relative_change_pct(
            frame[f'DACK__{axis}__최근3_이전9_로그차이']
        )

    result = add_local_shap_top10(frame, result, bundle)
    return result.sort_values(
        ['risk_rank_all_3372', '법인ID'], kind='mergesort'
    ).reset_index(drop=True)


web_scores = score_all_firms_202512(operating, bundle)
web_scores.to_csv(WEB_SCORE_PATH, index=False, encoding='utf-8-sig')
print(f'output={WEB_SCORE_PATH}')
print(f'rows={len(web_scores):,}')
print(f'unique_firms={web_scores["법인ID"].nunique():,}')
web_scores.head(10)

In [ ]:
assert len(web_scores) == 3372
assert web_scores['법인ID'].nunique() == 3372
assert web_scores['cutoff_month'].eq(202512).all()
assert np.isfinite(web_scores['risk_probability']).all()
assert web_scores['risk_rank_all_3372'].min() == 1
assert web_scores['risk_rank_all_3372'].max() == 3372

web_scores[[
    'risk_probability',
    'risk_probability_pct',
    'risk_rank_all_3372',
    'risk_percentile',
]].describe(percentiles=[0.5, 0.9, 0.95, 0.99])